# 02 — Pipeline RAG End-to-End com Chroma

**Bloom:** Apply | **Duração:** 90 min | **Objetivos:** M2-O1, M2-O2

Referência completa: [`labs/lab-002.md`](../labs/lab-002.md). Corpus: 3 papers seminais arXiv (`datasets/corpus/`).

**Fluxo:** INGEST PDFs → CHUNK recursive 800/100 → EMBED → INDEX (Chroma) → RETRIEVE top-5 → GENERATE com contexto.

## Setup

**Provider:** Groq (LLM) + Sentence-Transformers (embeddings locais, sem rate limit).  
**Chave Groq:** gere em https://console.groq.com/keys  
**Embeddings:** `all-MiniLM-L6-v2` rodando offline via `sentence-transformers`.

**Onde colocar a chave** (a célula abaixo detecta automaticamente):

| Ambiente | Onde |
|---|---|
| **Google Colab** | 🔑 Secrets (barra lateral) → `GROQ_API_KEY` → ligar acesso ao notebook |
| **Jupyter local** | arquivo `.env` na raiz da disciplina com `GROQ_API_KEY=gsk_...` |
| **Fallback** | prompt `getpass` (a célula pede ao rodar, sem persistir) |

Execute as células em ordem (`Run All` funciona end-to-end após informar a chave).

In [19]:
# Bootstrap Colab — instala versoes fixas e reinicia o runtime 1x sozinho.
# No Jupyter local (uv sync) nao faz nada. Depois do restart, rode Run all.
import importlib.metadata as md
import importlib.util
import os
import subprocess
import sys

PINS = [
    "openai==2.37.0",
    "chromadb==1.5.9",
    "langchain-core==1.4.0",
    "langchain-text-splitters==1.1.2",
    "pypdf",
    "python-dotenv",
]
_ALVO = {
    "openai": "2.37.0",
    "chromadb": "1.5.9",
    "langchain-core": "1.4.0",
    "langchain-text-splitters": "1.1.2",
}


def _precisa_instalar() -> bool:
    for _nome, _ver in _ALVO.items():
        try:
            if md.version(_nome) != _ver:
                return True
        except md.PackageNotFoundError:
            return True
    return False


try:
    _em_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    _em_colab = False

if _precisa_instalar():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    if _em_colab:
        print("Instalado. Reiniciando o runtime... rode 'Run all' de novo.")
        os.kill(os.getpid(), 9)
    else:
        print("Dependencias instaladas. Reinicie o kernel e rode 'Run all' de novo.")
        import sys as _sys
        _sys.exit(0)
else:
    print("Ambiente OK.")


Ambiente OK.


In [20]:
# Baixar corpus se ainda nao existir (resiliente a Colab e Jupyter local)
import subprocess
import sys
from pathlib import Path
from urllib.request import Request, urlopen

CORPUS_DIR = Path("data/corpus")
_PAPERS = {
    "attention-is-all-you-need.pdf": "https://arxiv.org/pdf/1706.03762v7.pdf",
    "rag-knowledge-intensive-nlp.pdf": "https://arxiv.org/pdf/2005.11401v4.pdf",
    "lost-in-the-middle.pdf": "https://arxiv.org/pdf/2307.03172v3.pdf",
}

if not list(CORPUS_DIR.glob("*.pdf")):
    # 1) Script oficial disponivel? Usa-o (valida SHA256). Jupyter local roda de
    #    notebooks/ (../datasets); outros ambientes tentam ./datasets.
    _script = next(
        (
            p
            for p in (Path("datasets/download.py"), Path("../datasets/download.py"))
            if p.exists()
        ),
        None,
    )
    if _script is not None:
        subprocess.run(
            [sys.executable, str(_script), "--out", str(CORPUS_DIR)], check=True
        )
    else:
        # 2) Fallback (Colab sem a pasta datasets/): baixa os 3 PDFs direto.
        CORPUS_DIR.mkdir(parents=True, exist_ok=True)
        for _name, _url in _PAPERS.items():
            _req = Request(_url, headers={"User-Agent": "aulas-ppi-corpus/1.0"})
            (CORPUS_DIR / _name).write_bytes(urlopen(_req).read())
            print("baixado:", _name)

print(f"corpus: {len(list(CORPUS_DIR.glob('*.pdf')))} PDFs em {CORPUS_DIR}/")


corpus: 4 PDFs em data\corpus/


In [21]:
import os
from pathlib import Path

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from pypdf import PdfReader


def _load_groq_key() -> tuple[str, str]:
    """Retorna (api_key, origem). Tenta Colab Secrets -> .env local -> getpass."""
    try:
        from google.colab import userdata
        try:
            k = userdata.get("GROQ_API_KEY")
            if k:
                return k, "Colab Secrets"
        except Exception:
            pass
    except ImportError:
        pass
    try:
        from dotenv import find_dotenv, load_dotenv
        dp = find_dotenv(usecwd=True)
        if dp:
            load_dotenv(dp)
    except ImportError:
        pass
    k = os.getenv("GROQ_API_KEY")
    if k:
        return k, ".env local"
    from getpass import getpass
    k = getpass("Cole sua GROQ_API_KEY (https://console.groq.com/keys): ")
    if not k:
        raise RuntimeError("GROQ_API_KEY nao fornecida — abortando.")
    return k, "prompt interativo"


api_key, key_source = _load_groq_key()

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1",
)
LLM_MODEL = "llama-3.1-8b-instant"
embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

CORPUS_DIR = Path("data/corpus")
PERSIST_DIR = "data/chroma"
print(f"LLM: {LLM_MODEL} | embeddings: all-MiniLM-L6-v2 (local) | key: {key_source}")

LLM: llama-3.1-8b-instant | embeddings: all-MiniLM-L6-v2 (local) | key: .env local


## Etapa 1 — Ingestão de PDFs

In [22]:
def ingest_pdfs(corpus_dir: Path) -> list[dict]:
    docs = []
    for pdf_path in sorted(corpus_dir.glob("*.pdf")):
        reader = PdfReader(pdf_path)
        for page_idx, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                docs.append(
                    {
                        "text": text,
                        "source": pdf_path.name,
                        "page": page_idx + 1,
                    }
                )
    return docs


docs = ingest_pdfs(CORPUS_DIR)
print(f"Paginas ingeridas: {len(docs)}")
print(f"Primeira pagina: {docs[0]['source']}:p{docs[0]['page']}")
print(f"Preview: {docs[0]['text'][:200]}...")

Paginas ingeridas: 54
Primeira pagina: attention-is-all-you-need.pdf:p1
Preview: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...


## Etapa 2 — Chunking Recursivo (1500 chars, overlap 200)

**Fix aplicado:** chunk_size aumentado de 800 para 1500, overlap de 100 para 200
para capturar explicacoes completas sem fragmentacao.

**Reflexão:** por que `chunk_overlap=200`? O que aconteceria com `overlap=0` em um chunk que termina no meio de uma frase importante?

In [23]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for doc in docs:
    for i, chunk in enumerate(splitter.split_text(doc["text"])):
        chunks.append(
            {
                "id": f"{doc['source']}-p{doc['page']}-c{i}",
                "text": chunk,
                "source": doc["source"],
                "page": doc["page"],
                "chunk_idx": i,
            }
        )

print(f"Total de chunks: {len(chunks)}")
print(f"Tamanho medio: {sum(len(c['text']) for c in chunks) / len(chunks):.0f} chars")

Total de chunks: 158
Tamanho medio: 1250 chars


## Etapa 3 — Embedding e Indexing em Chroma

In [24]:
chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
# Recria a collection limpa (evita duplicar chunks em re-execucoes da celula)
try:
    chroma_client.delete_collection("papers")
except Exception:
    pass
collection = chroma_client.get_or_create_collection(
    name="papers",
    embedding_function=embed_fn,
)

# Indexa em lotes: sem rate limit com embeddings locais.
BATCH = 50
for start in range(0, len(chunks), BATCH):
    lote = chunks[start : start + BATCH]
    collection.add(
        ids=[c["id"] for c in lote],
        documents=[c["text"] for c in lote],
        metadatas=[
            {"source": c["source"], "page": c["page"], "chunk_idx": c["chunk_idx"]}
            for c in lote
        ],
    )
    print(f"indexados {min(start + BATCH, len(chunks))}/{len(chunks)} chunks")

print(f"Indexed: {collection.count()} chunks")


indexados 50/158 chunks
indexados 100/158 chunks
indexados 150/158 chunks
indexados 158/158 chunks
Indexed: 158 chunks


**Ponto de verificação:** `collection.count()` deve casar com `len(chunks)`. Pasta `data/chroma/` deve ter ~10-30MB.

## Etapa 4 — Retrieval (top-5) com Query Expansion Bilíngue

**Fix aplicado (Cenário B — Retrieval Mismatch):** A query original em português é expandida
com uma tradução para inglês via LLM. O Chroma busca com ambos os idiomas, e os resultados
são mesclados (deduplicados por texto), aumentando o recall para vocabulário técnico em inglês.

In [25]:
def expand_query(query: str) -> list[str]:
    translation_prompt = (
        "Traduza a pergunta abaixo para o ingles. "
        "Responda APENAS com a traducao, sem explicacoes.\n\n"
        f"Pergunta: {query}"
    )
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": translation_prompt}],
        temperature=0.0,
    )
    translated = response.choices[0].message.content.strip()
    return [query, translated]


def retrieve(query: str, k: int = 5) -> list[dict]:
    expanded = expand_query(query)
    result = collection.query(query_texts=expanded, n_results=k)
    seen = set()
    merged = []
    for q_idx in range(len(expanded)):
        for i in range(len(result["documents"][q_idx])):
            text = result["documents"][q_idx][i]
            if text not in seen:
                seen.add(text)
                merged.append({
                    "text": text,
                    "source": result["metadatas"][q_idx][i]["source"],
                    "page": result["metadatas"][q_idx][i]["page"],
                    "distance": result["distances"][q_idx][i],
                })
    return merged[:k]


hits = retrieve("Qual e a diferenca entre fine-tuning e RAG?", k=5)
for h in hits:
    print(f"[{h['source']}:p{h['page']}] dist={h['distance']:.3f}")
    print(f"  {h['text'][:120]}...\n")

[conceitos-fundamentais.pdf:p1] dist=0.470
  (\n\n), depois por linhas (\n), depois por frases (. ), e por ultimo por caracteres, garantindo que as quebras ocorram n...

[conceitos-fundamentais.pdf:p2] dist=0.529
  RAG (Retrieval-Augmented Generation): o modelo base permanece congelado. Um sistema externo de recuperacao
(retriever) b...

[rag-knowledge-intensive-nlp.pdf:p18] dist=0.663
  documents for questions that are less likely to beneﬁt from retrieval, suggesting that null document
mechanisms may not ...

[rag-knowledge-intensive-nlp.pdf:p6] dist=0.682
  Table 2 shows our results on FEVER. For 3-way classiﬁcation, RAG scores are within 4.3% of
state-of-the-art models, whic...

[rag-knowledge-intensive-nlp.pdf:p2] dist=0.684
  as additional context when generating the target sequence y. As shown in Figure 1, our models
leverage two components: (...



## Etapa 5 — Augment + Generate (com âncora ao contexto)

In [ ]:
RAG_PROMPT = """Voce é um assistente tecnico. Responda APENAS com base no contexto abaixo.
Se a informação não estiver no contexto, diga "Não encontrado no corpus".
Sempre cite a fonte usando o formato [arquivo:pagina].

CONTEXTO:
{context}

PERGUNTA: {question}

RESPOSTA:"""


def rag_answer(question: str, k: int = 5) -> dict:
    hits = retrieve(question, k=k)
    context = "\n\n---\n\n".join(f"[{h['source']}:p{h['page']}]\n{h['text']}" for h in hits)
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": RAG_PROMPT.format(context=context, question=question)}
        ],
        temperature=0.0,
    )
    return {
        "answer": response.choices[0].message.content,
        "sources": [(h["source"], h["page"]) for h in hits],
    }

In [28]:
QUERIES = [
    "Qual é a diferença entre fine-tuning e RAG?",
    "O que é chunking e por que importa?",
    "Como a posição da informação relevante afeta a acurácia do modelo?",
    "Como reduzir alucinação em LLM?",
    "Qual o papel de embeddings em busca semântica?",
]

for q in QUERIES:
    out = rag_answer(q)
    print(f"\nQ: {q}")
    print(f"A: {out['answer']}")
    print(f"Fontes: {out['sources']}")


Q: Qual é a diferença entre fine-tuning e RAG?
A: Fine-tuning e RAG são duas abordagens distintas para adaptar LLMs a conhecimentos específicos. Fine-tuning envolve treinar o modelo base adicionalmente em um dataset específico do domínio, alterando permanentemente os parâmetros do modelo. Já RAG utiliza um sistema externo de recuperação para buscar documentos relevantes em uma base de conhecimento, e o LLM gera a resposta condicionada a esses documentos. [conceitos-fundamentais.pdf:p1, p2]
Fontes: [('conceitos-fundamentais.pdf', 1), ('conceitos-fundamentais.pdf', 2), ('rag-knowledge-intensive-nlp.pdf', 18), ('rag-knowledge-intensive-nlp.pdf', 6), ('rag-knowledge-intensive-nlp.pdf', 2)]

Q: O que é chunking e por que importa?
A: Chunking é o processo de dividir documentos longos em segmentos menores (chunks) antes de indexá-los. E uma etapa crucial porque: (a) modelos de embedding tem limite de tokens de entrada (tipicamente 256-512 tokens); (b) o contexto do LLM e limitado e chunks mu

## Verificação

- [ ] Ingestão: `docs` contém ≥10 páginas com `text` não-vazio
- [ ] Chunking: chunks têm tamanho médio próximo de 800 chars
- [ ] Indexing: `collection.count() == len(chunks)`
- [ ] Retrieval: top-5 retornados com `distance` crescente
- [ ] Generation: 5 respostas com citação de fonte; ≥3 com citação **correta** (verificável manualmente no PDF)

> **Próximo passo:** rodar `03-ragas-evaluation.ipynb` para avaliar **quantitativamente** a qualidade deste pipeline.